# Polypharmacy Risk Detection — Model Performance

Evaluates the **harmfulness head** of the Drug Interaction GNN on the held-out test set.
This notebook measures only the model's ability to classify drug combinations as polypharmacy risks — **not** side-effect prediction.

In [1]:
import sys
import json
import numpy as np
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score,
    roc_curve, precision_recall_curve, confusion_matrix,
)

sys.path.insert(0, ".")
from src.model import DrugInteractionGNN

print("Imports OK")

/home/satyam-rana/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
PROC = "data/processed"
MAP  = "data/mappings"

drug_features = np.load(f"{PROC}/drug_features.npy")
harm_labels   = np.load(f"{PROC}/harm_labels.npy")

edges = {
    "full": torch.load(f"{PROC}/full_edge_index.pt", weights_only=True),
    "dd":   torch.load(f"{PROC}/dd_edge_index.pt",   weights_only=True),
    "dp":   torch.load(f"{PROC}/dp_edge_index.pt",   weights_only=True),
    "pp":   torch.load(f"{PROC}/pp_edge_index.pt",   weights_only=True),
}

splits    = np.load(f"{PROC}/splits.npz")
test_idx  = splits["test_idx"]
val_idx   = splits["val_idx"]

with open(f"{MAP}/drug_to_idx.json")      as f: drug_to_idx      = json.load(f)
with open(f"{MAP}/drug_pair_to_idx.json") as f: drug_pair_to_idx = json.load(f)
with open(f"{MAP}/protein_to_idx.json")   as f: protein_to_idx   = json.load(f)

num_drugs    = len(drug_to_idx)
num_proteins = len(protein_to_idx)

print(f"Drugs: {num_drugs} | Proteins: {num_proteins}")
print(f"Test pairs: {len(test_idx)} | Harmful: {int(harm_labels[test_idx].sum())} ({harm_labels[test_idx].mean()*100:.1f}%)")

Drugs: 2135 | Proteins: 19122
Test pairs: 6348 | Harmful: 1918 (30.2%)


In [3]:
def build_pair_tensor(drug_pair_to_idx, drug_to_idx, device):
    pairs_sorted = sorted(drug_pair_to_idx.items(), key=lambda x: x[1])
    rows = []
    for key, _ in pairs_sorted:
        d1, d2 = key.split("|")
        rows.append([drug_to_idx.get(d1, 0), drug_to_idx.get(d2, 0)])
    return torch.tensor(rows, dtype=torch.long, device=device)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

pair_tensor = build_pair_tensor(drug_pair_to_idx, drug_to_idx, device)

Device: cuda


In [4]:
model = DrugInteractionGNN(
    drug_feature_dim = drug_features.shape[1],
    num_proteins     = num_proteins,
    num_side_effects = sp.load_npz(f"{PROC}/label_matrix.npz").shape[1],
).to(device)

model.load_state_dict(torch.load("models/best_model.pt", map_location=device, weights_only=True))
model.eval()
print(f"Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters")

Model loaded — 2,643,841 parameters


In [5]:
drug_feat_t  = torch.tensor(drug_features, dtype=torch.float32, device=device)
edge_index_t = edges["full"].to(device)
dd_edge_t    = edges["dd"].to(device)
dp_edge_t    = edges["dp"].to(device)
pp_edge_t    = edges["pp"].to(device)

BATCH = 512

def run_harm_inference(idx_arr):
    """Returns (probs, true_labels) for a set of pair indices."""
    all_probs = []
    with torch.no_grad():
        z = model.encode(drug_feat_t, edge_index_t, num_drugs,
                         dd_edge_t, dp_edge_t, pp_edge_t)
        for start in range(0, len(idx_arr), BATCH):
            batch = idx_arr[start: start + BATCH]
            se_logits   = model.decode_side_effects(z, pair_tensor[batch])
            se_probs    = torch.sigmoid(se_logits)
            harm_logits = model.decode_harmfulness(se_probs)
            harm_probs  = torch.sigmoid(harm_logits).squeeze(1).cpu().numpy()
            all_probs.append(harm_probs)
    return np.concatenate(all_probs), harm_labels[idx_arr]

test_probs, test_true = run_harm_inference(test_idx)
val_probs,  val_true  = run_harm_inference(val_idx)

print(f"Inference done — test={len(test_probs)} | val={len(val_probs)}")

Inference done — test=6348 | val=6347


In [6]:
THRESHOLD = 0.5
test_preds = (test_probs >= THRESHOLD).astype(int)

metrics = {
    "ROC-AUC":         roc_auc_score(test_true, test_probs),
    "Avg Precision":   average_precision_score(test_true, test_probs),
    "F1 Score":        f1_score(test_true, test_preds),
    "Accuracy":        accuracy_score(test_true, test_preds),
    "Precision":       precision_score(test_true, test_preds),
    "Recall":          recall_score(test_true, test_preds),
}

print("="*46)
print(f"  {'Metric':<22} {'Test Set':>10}")
print("="*46)
for k, v in metrics.items():
    print(f"  {k:<22} {v:>10.4f}")
print("="*46)
print(f"  Val ROC-AUC (ref)      {roc_auc_score(val_true, val_probs):>10.4f}")

  Metric                   Test Set
  ROC-AUC                    0.8838
  Avg Precision              0.7473
  F1 Score                   0.5307
  Accuracy                   0.4723
  Precision                  0.3628
  Recall                     0.9875
  Val ROC-AUC (ref)          0.8777
